# Assessing Bicycle Lane Quality Using Smartphone Sensor Data

This notebook contains the complete data analysis pipeline for
classifying bicycle-lane sections as Smooth or Bumpy using
smartphone sensor data.

The analysis includes data loading, preprocessing, feature
extraction, feature selection, supervised and unsupervised
learning, model evaluation, and deployment.

## 0. Environment and Reproducibility
The Python environment and package versions are recorded below to
support reproducibility.

In [4]:
import sys

import pandas as pd
import numpy as np
import sklearn 
import matplotlib as plt

print("Python version:", sys.version)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib:", plt.__version__)

Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
pandas: 2.2.3
numpy: 2.3.5
scikit-learn: 1.9.0
matplotlib: 3.10.0


In [5]:
#Standard library only
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering,
    DBSCAN
)
from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

## Data Collection Metadata

The dataset was collected using smartphone sensor recordings while cycling
on smooth and bumpy bicycle lanes.

The following metadata documents the phone used, phone placement, bicycle,
and route characteristics for each recording.

In [6]:
Collection_info = {'P1':{'Phone':'iphone 15', 'placement':'Vertical on bicycle basket'},
                   'P2':{'Phone':'iphone 13', 'placement':'Vertical on bicycle basket'},
                   'P3':{'Phone':'iphone 17 pro max', 'placement':'Vertical on bicycle basket'}}

Route_info = {'P1_smooth': 'Near Geldropsedijk bus stop','P1_bumpy': 'Eisenhowerlaan',
    'P2_smooth': '','P2_bumpy': '',
    'P3_smooth': '','P3_bumpy': ''}



In [7]:
def load_and_merge(acc_path, grav_path, gyro_path):
    """
    Load accelerometer, gravity, and gyroscope data and combine
    them using their timestamps.
    """

    acc = pd.read_csv(acc_path).rename(
        columns={'x': 'acc_x', 'y': 'acc_y', 'z': 'acc_z'}
    )

    grav = pd.read_csv(grav_path).rename(
        columns={'x': 'grav_x', 'y': 'grav_y', 'z': 'grav_z'}
    )

    gyro = pd.read_csv(gyro_path).rename(
        columns={'x': 'gyro_x', 'y': 'gyro_y', 'z': 'gyro_z'}
    )

    # Check that all recordings have the same number of observations
    if not (len(acc) == len(grav) == len(gyro)):
        raise ValueError("Sensor files have different numbers of rows.")

    # Check timestamp alignment
    if not acc['time'].equals(grav['time']):
        raise ValueError("Accelerometer and gravity timestamps are not aligned.")

    if not acc['time'].equals(gyro['time']):
        raise ValueError("Accelerometer and gyroscope timestamps are not aligned.")

    # Combine the sensor measurements
    df = acc[
        ['time', 'seconds_elapsed', 'acc_x', 'acc_y', 'acc_z']
    ].copy()

    df[['grav_x', 'grav_y', 'grav_z']] = grav[
        ['grav_x', 'grav_y', 'grav_z']
    ]

    df[['gyro_x', 'gyro_y', 'gyro_z']] = gyro[
        ['gyro_x', 'gyro_y', 'gyro_z']
    ]

    return df


In [8]:
# Define the location of each participant's recordings

recordings = {
    'P1_smooth': {
        'label': 'smooth',
        'folder': '../data_raw/Smooth/P1'
    },
    'P1_bumpy': {
        'label': 'bumpy',
        'folder': '../data_raw/Bumpy/P1'
    },
    'P2_smooth': {
        'label': 'smooth',
        'folder': '../data_raw/Smooth/P2'
    },
    'P2_bumpy': {
        'label': 'bumpy',
        'folder': '../data_raw/Bumpy/P2'
    },
    'P3_smooth': {
        'label': 'smooth',
        'folder': '../data_raw/Smooth/P3'
    },
    'P3_bumpy': {
        'label': 'bumpy',
        'folder': '../data_raw/Bumpy/P3'
    }
}

In [10]:
merged_recordings = {}

for name, recording in recordings.items():

    folder = recording['folder']

    df = load_and_merge(
        f"{folder}/Accelerometer.csv",
        f"{folder}/Gravity.csv",
        f"{folder}/Gyroscope.csv"
    )

    merged_recordings[name] = df

    print(
        f"{name}: "
        f"{len(df)} rows, "
        f"{df['seconds_elapsed'].max():.1f} seconds"
    )

FileNotFoundError: [Errno 2] No such file or directory: '../data_raw/Smooth/P1/Accelerometer.csv'

In [9]:
merge_recordings = {}

for name, recording in recordings.items():

    folder = recording['folder']

    df = load_and_merge(
        f"{folder}/Accelerometer.csv",
        f"{folder}/Gravity.csv",
        f"{folder}/Gyroscope.csv")

    merge_recordings[name] = df

    print(
        f"{name}: "
        f"{len(df)} rows, "
        f"{df['seconds_elapsed'].max():.1f} seconds"
    )

FileNotFoundError: [Errno 2] No such file or directory: '../data_raw/Smooth/P1/Accelerometer.csv'